1. IMPORT lIBRARY

In [1]:
!pip install pandas numpy Sastrawi scikit-learn nltk

You should consider upgrading via the '/Users/irma95/.pyenv/versions/3.10.0/bin/python3.10 -m pip install --upgrade pip' command.


In [2]:
# Import library
import pandas as pd
import numpy as np
import re
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/irma95/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/irma95/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/irma95/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
pd.set_option('display.max_rows', None)      # tampilkan semua baris
pd.set_option('display.max_columns', None)   # tampilkan semua kolom
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

2. Data Preparation

In [4]:
df = pd.read_csv('../data/SP4N LAPOR.csv')


In [5]:
df.head()

,userName,score,at,content
0,kevhin sandra,5,2024-09-06 05:26:43,Mantab
1,rama noer,1,2024-09-06 01:59:06,"Disuruh ubah pasword maksa banget, lupa pasword ngak terjadi apa"""
2,Andi Saputro,1,2024-09-05 07:39:07,Aplikasinya untuk apa ya . Buat laporan dr juli blm direspon . !
3,nʎɐq bay,1,2024-09-04 01:54:30,"APLIKASI GAK BERGUNA, MASIH LOGIN SAJA SUDAH ERROR. GAK BISA DIPAKE. ITULAH HEBATNYA PEMERINTAH KITA. GAJI DAN FASILITAS MEWAH KERJA NOL."
4,Juli Syahputra,2,2024-09-02 13:14:57,Laporan saya tidak pernah di proses


In [6]:
df.sample(5)

,userName,score,at,content
63,service tasik,3,2024-07-13 23:31:40,jelek juga nih aplikasinya layar edit terhalang keyboard...mau ngirim error gbr hrs d bawah 2 mb dst coba d update perbaiki
349,Ryan Airlangga (Rey),1,2023-05-30 20:48:04,Internal server error...
588,Rizal Aji Pamungkas,2,2022-04-25 07:17:58,"sejauh ini baik, tapi tiba tiba terus minta mengisi biodata padahal sebelumnya sudah mengisi dan membuat laporan, mau saya tanggapi dan balas komentar dari tanggapan laporan saya tapi terus stuck dan berhenti di pengisian biodata terus. tolong perbaiki"
1070,Febrina Wonosantoso,1,2020-09-18 10:09:04,Percuma lapor klw ga direspon. Sudah 2 minggu lapor ga ada respon juga. Mengecewakan.
439,PUTRA WALDI S,1,2022-11-30 06:47:02,Laporan sudah sesuai syarat dan ketentuan.sudah lama sekali saya menunggu tidak ada tindak lanjut dan solusi.


In [7]:
df.tail()

,userName,score,at,content
1620,Pengguna Google,5,2019-09-13 04:23:38,"Jadi semakin mudah utk mengawasi jalannya pemerintahan, ayo jangan ragu download aplikasi ini agar bisa lapor kapan aja dimana aja."
1621,Pengguna Google,5,2019-09-12 06:31:32,Lebih memudah kan dan gk bikin bingung 😘😘😘😘
1622,Pengguna Google,3,2019-09-11 00:57:09,Kucoba baru komen
1623,Pengguna Google,5,2019-09-10 05:48:00,Aplikasinya bagus. Semoga bisa meningkatkan kualitas pelayanan publik
1624,Pengguna Google,5,2019-09-10 04:40:12,Sangat membantu dalam menangani pelayanan publik


In [8]:
# hapus missing value
df.isnull().sum()

userName    0
score       0
at          0
content     0
dtype: int64

tidak ada missing value

In [9]:
# hapus data duplikat
df.duplicated().sum()

np.int64(0)

tidak ada data duplikat

In [10]:
# rubah tipe data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1625 entries, 0 to 1624
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   userName  1625 non-null   object
 1   score     1625 non-null   int64 
 2   at        1625 non-null   object
 3   content   1625 non-null   object
dtypes: int64(1), object(3)
memory usage: 50.9+ KB


In [11]:
df['at'] = pd.to_datetime(df['at'])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1625 entries, 0 to 1624
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   userName  1625 non-null   object        
 1   score     1625 non-null   int64         
 2   at        1625 non-null   datetime64[ns]
 3   content   1625 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 50.9+ KB


In [13]:
# hapus kolom userName
df = df.drop(columns=['userName'])


In [14]:
df.columns

Index(['score', 'at', 'content'], dtype='object')

3. DATA PREPROCESSING

1. Casefolding : fungsi untuk mengubah text menjadi lowercase.

In [15]:

def casefoldingText(text):
    text = str(text).lower()
    return text

2. Cleaning Text : fungsi untuk menghapus mentions, hastag, RT, link, dan numbers dari text.

In [16]:


def cleaningText(text):
    text = re.sub(r"http\S+", "", text)        # hapus url
    text = re.sub(r"@\w+", "", text)           # hapus mention
    text = re.sub(r"#\w+", "", text)           # hapus hashtag
    text = re.sub(r"\bRT\b", "", text)         # hapus RT
    text = re.sub(r"\n", " ", text)            # hapus newline
    text = re.sub(r"[^a-zA-Z\s]", "", text)    # hapus simbol & angka
    text = re.sub(r'(.)\1{2,}', r'\1', text)   # normalisasi huruf
    text = re.sub(r'\b(wk|wkwk|haha|hehe|hihi)+\b', "", text)
    text = re.sub(r"\s+", " ", text)           # rapikan spasi
    text = text.strip()
    text = re.sub(r'\bx+\b', '', text, flags=re.IGNORECASE)
    return text

3. Normalization : mengubah kata slang/tidak baku menjadi kata baku

In [31]:
slangwords = {
    # negasi
    "ga":"tidak",
    "gak":"tidak",
    "gk":"tidak",
    "ngga":"tidak",
    "nggak":"tidak",
    "tdk":"tidak",
    "kaga" : "tidak",
    "gaada" : "tidak-ada",
    "lanjutannya" :"kabar",

    # waktu / aspek
    "udh":"sudah",
    "udah":"sudah",
    "sdh":"sudah",
    "blm":"belum",
    "belom":"belum",
    "dlu":"dulu",
    "skrg":"sekarang",
    "lg":"lagi",
    "lgi":"lagi",
    "sampe":"sampai",

    # kata ganti
    "gw":"saya",
    "gua":"saya",
    "sy":"saya",
    "sya":"saya",
    "lu":"kamu",
    "ku":"aku",

    # preposisi
    "dr":"dari",
    "dri":"dari",
    "dgn":"dengan",
    "utk":"untuk",
    "dlm":"dalam",

    # konjungsi
    "tp":"tapi",
    "tpi":"tapi",
    "krn":"karena",

    # intensifier
    "bgt":"banget",
    "bngt":"banget",

    # aplikasi
    "apk":"aplikasi",
    "app":"aplikasi",
    "apps":"aplikasi",
    "apknya":"aplikasi",

    # verba umum
    "dpt":"dapat",
    "jd":"jadi",
    "liat":"lihat",
    "trus":"terus",
    "mulu":"terus",
    "nyari":"mencari",
    "ngelamar":"melamar",
    "nglamar":"melamar",
    "loker" : "lowongan",
    "terima kasih" : "terimakasih",
    "makasih" : "terimakasih",
    "thanks" : "terimakasih",
    "thx" : "terimakasih",
    "thank" : "terimakasih",
    "thnk" : "terimakasih",
    "kasih" : "beri",
    "info" : "informasi",


    # demonstratif
    "ni":"ini",
    "nih":"ini",
    "yaa":"ya",
    "yah":"ya",

    # lain-lain
    "jg":"juga",
    "bs":"bisa",
    "org":"orang",
    "dll":"dan lain lain",
    "ko":"kok",
    "eror":"error",
    "bnyk":"banyak",
    "bnyak":"banyak",
    "nipu":"menipu",
    "tipu":"menipu",
    "hrd":"hrd",
    "wa":"whatsapp",
    "hp":"handphone",
    "pt" : "perusahaan",
    "updateannya" : "update",
    "ilang": "hilang",
    "ttp" : "tetap",
    "pekerjaannya" : "pekerjaan",
    "mudahan" : "semoga",
    "dpet" : "mendapatkan",
    "bagu" : "bagus",
    "yg" : "yang",
    "cukup": "cukup",
    "baik" : "baik"
}


In [32]:
def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

4. Tokenizing Text : fungsi untuk memecah text menjadi list tokens

In [33]:
def tokenizingText(text): # Tokenizing or splitting a string, text into a list of tokens
    text = word_tokenize(text)
    return text

5. Stopword Removal : fungsi untuk menghapus stopwords dalam bahasa Indonesia ataupun Inggris.

In [34]:

def remove_stopwords(text):  # Remove stopwords in a text
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)

    listStopwords.update([
        'iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah",
        "woi","woii","woy","jobstreet","linkedin","glints","kalibrr","indeed",
        'pt','perusahaan','cv','tbk',"semua","sama","sekarang","lagi","mungkin",
        "tenang","saja","masih","akan","haduhh","lucu","wkwkwkwk",
        "jobsteet","asw","asyudahlah","lina","trs"
    ])


    listStopwords.discard("terimakasih")
    listStopwords.discard("dapat")
    listStopwords.discard("mendapatkan")
    listStopwords.discard("kasih")


    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)

    return filtered


6. Stemming : fungsi untuk mengubah text menjadi bentuk dasar. Namun pada proyek ini tidak akan digunakan karena proses nya yang lama.

In [21]:
#factory = StemmerFactory()
#stemmer = factory.create_stemmer()

#def stemmingText(tokens):
    #return [stemmer.stem(word) for word in tokens]

Catatan : Stemming tidak di gunakan pada proses ini karena lama

7. toSentence

In [35]:

def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

In [36]:
# mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'casefolding_text'
df['casefolding_text'] = df['content'].apply(casefoldingText)

# membersihkan teks dan menyimpan ya di kolom 'clean_content'
df['cleaning_text'] = df['casefolding_text'].apply(cleaningText)

# mengganti kata-kata slang/tidak baku menjadi kata baku
df['slangword_text'] = df['cleaning_text'].apply(fix_slangwords)

# memecah teks menjadi token (kata-kata) dan menyimpanya di 'tokenizing_text'
df['tokenizing_text'] = df['slangword_text'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'stopword_text'
df['stopword_text'] = df['tokenizing_text'].apply(remove_stopwords)

# menggunakan stemming untuk kata dasar'
# df_clean['stemming_text'] = df_clean['stopword_text'].apply(stemmingText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'final_text'
df['final_text'] = df['stopword_text'].apply(toSentence)


In [38]:
df.sample()

,score,at,content,casefolding_text,cleaning_text,slangword_text,tokenizing_text,stopword_text,final_text
818,1,2021-08-23 20:01:17,👎,👎,,,[],[],


In [37]:
df.iloc[1534]

score                                 5
at                  2019-09-26 04:19:06
content                      Cukup baik
casefolding_text             cukup baik
cleaning_text                cukup baik
slangword_text               cukup baik
tokenizing_text           [cukup, baik]
stopword_text                        []
final_text                             
Name: 1534, dtype: object